# Load All Your Imports:
This Section includes all the imports that are required to work along

In [ ]:
# Install the required libraries if not available
!pip install sentence_transformers
# For Grammer check
!pip install gingerit
# For Reading PDF and Writing a PDF
!pip install PyMuPDF

# For Basic Operation
import pandas as pd
import logging
import os
import sys
import io
import json
import copy
import time
import re
import queue
import threading

# For Reading PDF
import fitz

# For NLTK
import nltk

# To operate on Google Collab
from google.colab import files

from pathlib import Path

# NLTK Related
from sentence_transformers import SentenceTransformer, util
from nltk.tag import pos_tag
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
from nltk import pos_tag
from nltk.corpus import stopwords

# Grammer Related
from gingerit.gingerit import GingerIt

# For Json return 
from flask import jsonify

     |████████████████████████████████| 71kB 3.3MB/s 
     |████████████████████████████████| 1.8MB 5.2MB/s 
     |████████████████████████████████| 1.2MB 15.3MB/s 
     |████████████████████████████████| 2.9MB 22.8MB/s 
     |████████████████████████████████| 890kB 38.5MB/s 
  Created wheel for sentence-transformers: filename=sentence_transformers-0.4.1.2-cp36-none-any.whl size=103068 sha256=12c99623053745da9148e1542bbc5e3f8048aeb0a67daa9e9e5595ed1897020a
  Stored in directory: /root/.cache/pip/wheels/3d/33/d1/5703dd56199c09d4a1b41e0c07fb4e7765a84d787cbdc48ac3
  Created wheel for sacremoses: filename=sacremoses-0.0.43-cp36-none-any.whl size=893261 sha256=bad4f61d9dc9c79eb3f5a3e64cbcc945731a6d68ea8a3c35c9663d7982be4718
  Stored in directory: /root/.cache/pip/wheels/29/3c/fd/7ce5c3f0666dab31a50123635e6fb5e19ceb42ce38d4e58f45
Successfully built sentence-transformers sacremoses
     |████████████████████████████████| 6.3MB 4.7MB/s 
[nltk_data] Downloading package stopwords to /root/nltk_d

# Class CEFR
Gives information regarding the Part of Speech and CEFR Level for a Particular word 

In [ ]:
class Cefr:
    # Consisting of word + part of speech combination as key and cefr level as value
    word_pos_dict = dict()
    df = pd.read_csv(open('master_cefr.csv'))

    # Conversion map for part of speech
    conversion_map = {'DET': 'dt', 'VERB': 'v', 'NOUN': 'n', 'ADJ': 'aj', 'ADV': 'av', 'PRONOUN': 'pn', 'ADP': 'pp',
                      'CONJ': 'cj', 'NAN': 'na'}

    def __init__(self):
        # Read the potential csv file
        logging.debug("Data read success")

        for row_index in self.df.index:
            combine = self.df['word'][row_index].lower() + "_" + self.df['wor_pos_map'][row_index]
            self.word_pos_dict[combine] = self.df['cefr'][row_index]

    try:
        df['wor_pos_map'] = df['pos'].replace(conversion_map)

        def getCefr(self, word, part_of_speech):
            """
            Use this method in order to receive the CEFR level of the word and part of speech combination
            :param self: word
            :param word: word
            :param part_of_speech: part of speech of the word
            :return: reduced cefr level
            """
            word = str(word).lower()
            pos = str(part_of_speech).lower()
            logging.debug(f"Requesting for {word} with part of speech {pos} ")

            if word != 'no arg' and pos != 'no arg':
                combo = word + "_" + pos
                ref = Cefr.word_pos_dict.get(combo)
                return ref
            else:
                return None

        def pos_converter(self='', pos='no arg'):
            """
            Use this method to receive the acronym of the part of speech map
            :param: part of speech as received from the data file
            :return: converted acronym for the part of speech
            """
            return Cefr.conversion_map.get(pos.upper())

    except:
        logging.error("Read Error ", sys.exc_info())


# Synonym Retriver
This class is used to retrieve the list of Synonyms for a particular word 

In [ ]:
class SynRetriever:
  
    def __init__(self):
        # Retrieve the json data from both the files.
        try:
            # with open(path + '/Data Files/ThesaurusDict.json') as thes_json_file:
            self.thes_json = json.load(open('ThesaurusDict.json'))

            # with open(path + '/Data Files/SynonymsDict.json') as syn_json_file:
            self.syn_json = json.load(open('SynonymsDict.json'))
        except:
            print('Error in retrieving file')

    def retrieveSynsByPos(self, word, Pos):
        """Functions retrieves only synonyms for PoS from both sources
        Args:
            word (string): Word to be found
            Pos (string): Parts of speech of the word

        Returns:
            [string]: Returns the synonyms of the word
        """
        thes_list = self.thes_json.get(word + '_' + Pos)
        if thes_list is None:
            thes_list = []
        syn_list = self.syn_json.get(word)
        if syn_list is None:
            syn_list = []
        comb_list = thes_list
        if len(comb_list) > 0:
            comb_list = self.refactor_list(comb_list)

        return comb_list

    def retrieveAllSyns(self, word):
        """Function retrieves synonyms for all PoS's
        Args:
            word (string): Word to be found

        Returns:
            [string]: Returns all the synonyms of the word
        """
        possible_pos = ['n', 'v', 'av', 'na', 'aj', 'pp', 'pn']
        syn_list = self.syn_json.get(word)
        if syn_list is None:
            syn_list = []

        cumulative_list = list()
        for pos in possible_pos:
            pos_word = word + '_' + pos
            syns = self.thes_json.get(pos_word)
            if syns is not None:
                cumulative_list += syns

        if len(cumulative_list) > 0:
            cumulative_list = self.refactor_list(cumulative_list)

        return cumulative_list

    def refactor_list(self, oldList):
        """Method to refactor the lsit of duplicates and unwanted spaces.
        Args:
            oldList ([string]): List to be refactored.

        Returns:
            [string]: Refactored list.
        """
        # Get unique list from the combination.
        oldList = list(dict.fromkeys(oldList))
        # Removes spaces before words.
        return list(map(str.lstrip, oldList))

# Semantic Check For the Sentences
This class acts as a helper to retrieve the semantic score for a list of synonyms and a sentence that is fed in

In [ ]:
class SemanticCheck:
    # NLTK tag list -> custom tag for project
    pos_convert = {'NN': 'n', 'NNS': 'n', 'VB': 'v', 'VBG': 'v', 'VBD': 'v',
                   'VBN': 'v', 'VBP': 'v', 'VBZ': 'v', 'JJ': 'aj', 'JJR': 'aj', 'JJS': 'aj',
                   'RB': 'av', 'RBR': 'av', 'RBS': 'av', 'IN': 'pp', 'CC': 'cj'}

    def __init__(self):
        self.model = SentenceTransformer('roberta-base-nli-stsb-mean-tokens',device="cuda")

    def form_sentences(self, query, word, synonyms, index, seperator):
        """This method makes sentences for all the synonyms of the word provided.

        Args:
            query (string): The original sentence
            word (string): Word to be replaced
            synonyms ([string]): List of synonyms of the word to be replaced.
            index (int): Position of the word in the query
            seperator (string): Seperator for the combined sentences

        Returns:
            (string, [string]): A string of the merged sentences, list of individual sentences
        """
        words = query.replace("\n"," ").split(' ')
        sentences = list()
        
        for synonym in synonyms:
            word_list = copy.deepcopy(words)
            word_list[index] = synonym
            sentence = ''
            for word in word_list:
                sentence = sentence + ' ' + word
            sentences.append(sentence.strip())

        merged_sentences = seperator.join(sentences)
        return merged_sentences, sentences

    def checkSynPos(self, query, word, synonyms, index, req_pos):
        """Returns syns with same PoS as the original word.
        Args:
            query (string): The original sentence.
            word (string): Word to be replaced.
            synonyms ([string]): List of synonyms of the word to be replaced.
            index (int): Position of the word in the query.
            :param req_pos: part of speech we are looking for

        Returns:
            [string]: List of acceptable synonyms.
        """
        wordList = copy.deepcopy(synonyms)
        wordList.insert(0, word)
        merged_sentences, sentences = self.form_sentences(query, word, wordList, index, '\n')
        del sentences
        tagged_sent = pos_tag(merged_sentences.split())

        acceptable_syns = set()
        
        for tag in tagged_sent:
            if tag[0] in synonyms and self.pos_convert.get(tag[1]) == req_pos:
                acceptable_syns.add(tag[0])

        return acceptable_syns

    def calcSemanticScore(self, query, word, synonyms, index,part_os):
        """Calculates the semantic similarity for the sentences with syns substituted.
        Args:
            query (string): The original sentence
            word (string): Word to be replaced
            synonyms ([string]): List of synonyms of the word to be replaced.
            index (int): Position of the word in the query

        Returns:
            [tuple]: Returns a list of tuples of (word, similarity, sentence)
        """
        corpus = []
       
        for val in synonyms:
            li = query.split()
            li[index] = val
            corpus.append(" ".join(li))

        if len(corpus) == 1:
            corpus.append(query)

        paraphrases = util.paraphrase_mining(self.model, corpus, corpus_chunk_size=len(corpus), top_k=1)

        result_list = set()

        for scores, i, j in paraphrases:
            split_word = corpus[i].split()[index]
            result_list.add((scores, corpus[i].split()[index], corpus[i]))

        sorted_list = sorted(result_list,key=lambda x:x[0],reverse=True) 
        
        return sorted_list[0]

# Custom Priority Queue Class
This is a pojo class to store certain values that is used later on to generate the json structure

In [ ]:
class Word(object):
  def __init__(self,score,synonym_word,synonym_cefr,sentence,original_word):
    self.score = score
    self.synonym_word = synonym_word
    self.sentence = sentence
    self.original_word = original_word
    self.synonym_cefr = synonym_cefr
  
  def __lt__(self,other):
    return self.score > other.score
  
  def __gt__(self,other):
    return self.score < other.score

# Code to Read the PDF and solve the problem statement
Code to parse through the PDF file, reduce the CEFR level of all the eligible words and return back a json response along with a modified PDF

In [ ]:
class CustomReader:
    # TAG's to exclude as they are conjunctions or Pronouns or words like where,how,what etc
    pos_to_exclude = ['CC', 'CD', 'MD', 'NNP', 'NNPS', 'PDT', 'DT', 'PRP', 'PRP$', 'RP', 'TO', 'WDT', 'WP', 'WRB']

    # NLTK tag list -> custom tag for project
    pos_converter = {'NN': 'n', 'NNS': 'n', 'VB': 'v', 'VBG': 'v', 'VBD': 'v',
                     'VBN': 'v', 'VBP': 'v', 'VBZ': 'v', 'JJ': 'aj', 'JJR': 'aj', 'JJS': 'aj',
                     'RB': 'av', 'RBR': 'av', 'RBS': 'av', 'IN': 'pp', 'CC': 'cj'}

    # stop words - words that we don't want to include
    # 1. Subordinating Conjunction
    stop_words_subs = ['after', 'although', 'as if', 'as long as', 'as much as', 'as soon as', 'as though', 'because',
                       'before', 'by the time', 'even if', 'even though', 'if', 'if only', 'if then', 'if when',
                       'in as much',
                       'in order that', 'lest', 'now', 'now since', 'now that', 'now when', 'once', 'provided',
                       'provided that',
                       'rater than', 'since', 'so that', 'supposing', 'than', 'that', 'though', 'til', 'unless',
                       'until',
                       'when', 'where', 'whereas', 'whenever', 'wherever', 'where if', 'which', 'while', 'who',
                       'whoever', 'why']

    # 2. Co-ordinating Conjunction
    stop_words_conj = ['For', 'Nor', 'But', 'Or', 'Yet', 'So', "you're", "They're", "monthly", "month",
                       "yearly", "year","daily", "day"]

    '''
    Dictionary Structure : {word:([A_CEFR_LIST],[B_CEFR_LIST],[C_CEFR_LIST])}
    '''
    synonym_dict = {}

    '''
    Final set to refer for CEFR Level A, B, C {(word,score,sentence)} 
    Each set comprises of @max 3 words in each level 
    '''
    set_a = queue.PriorityQueue()
    set_b = queue.PriorityQueue()
    set_c = queue.PriorityQueue()

    '''
    Used as a class variable to initialise the initial highest 
    score @ Level 0 - {When sentences are scored during the first semantic check run}
    '''
    initial_highest_score = 0.00

    '''
    Number of levels the sentences can recurse through
    in order to find a perfect synonym of the lowest CEFR order 
    '''
    max_allowed_levels = 5

    '''
    When sentences are checked for grammer,
    the following list gets populated
    '''
    grammer_checked_list = set()

    '''
    List to maintain words Track as they recurse through
    multiple levels
    '''
    word_tracker = []

    def __init__(self, page_number=0, file_path_to_pdf="", stop_word_list=[stop_words_subs, stop_words_conj]):
        try:
            self.pdf_document = file_path_to_pdf
        except FileNotFoundError:
            print("File Not Found Error")
        except FileExistsError:
            print("File Does not Exist Error")
        else:
            # Open the PDF Document as doc
            self.doc = fitz.open(self.pdf_document)
            # Load Page based on Page number
            self.page = self.doc.loadPage(page_number)
            # Split the page as paragraphs
            self.page_block = self.page.getText("blocks")
            # Add conjunctions to the list of stop words - Note: Some conjunction
            # which are not in stopwords('english') are added here 
            list_of_stop_words = self.stop_words_conj
            # Extend the list of NLTK stop words
            list_of_stop_words.extend(stopwords.words("english"))
            self.stop_words = set(list_of_stop_words)
            # This parser is a Grammer tool helper - Helps in correcting grammatical mistakes
            self.parser = GingerIt()
            self.title = self.doc.metadata.get('title')
            self.author = self.doc.metadata.get('author')

    def iterate_over_paragraph(self, to_reduce=False, to_increase=False):
        '''
        Use this method once the page has been loaded
        1. Work on each paragraph
        2. Split paragraph as sentences
        3. Prepare the Synonyms list of all the eligible words
        4. Decrease or Increase the CEFR level of a word - In each sentence
        5. Grammer Check
        6. Return a json response
        '''
        if len(self.page_block) != 0:
            # List of (sentence,[word_to_replace,pos_tag])
            master_data = []
            # Position to indicate where each paragraph breaks
            # Contains (paragraph_no,line_no,number of words in the same)
            para_break = []

            for index, value in enumerate(self.page_block):
                # Start with the paragraph
                paragraph_to_operate = value[4]
                # Break the paragraph into lines
                initial_list = paragraph_to_operate.split('.')
                # print(f"Each line is - {initial_list}")
                list_without_breaks = []

                for line_no,sentence in enumerate(initial_list):
                  # To identify true 'period'
                  position_of_break = -1

                  flag = sentence.endswith('\n')
                  if flag and (sentence!=' ' or sentence!=''):
                      position_of_break = sentence.index("\n")

                  new_sentence = sentence.strip().replace("-\n", '').replace('\n', ' ').replace('\t',' ')
                  if new_sentence != ' ' or new_sentence != '':
                    if line_no == 0 and flag:
                      is_line_special = True
                    else:
                      is_line_special = False
                    para_break.append((index,line_no,len(new_sentence),is_line_special,position_of_break))

                    list_without_breaks.append((new_sentence,not flag))

                # print(f"Each refined line is - {list_without_breaks}")    
                # print(f"Parabreak - {para_break} ")

                # para_break.append(len(list(paragraph_to_operate.split())))
                '''
                Each sentence present in the list must be iterated and reduced
                '''
                for sentence,period_flag in list_without_breaks:
                    # Break the list into word Tokens
                    list_data = sentence.split()
                    
                    # Contains words which have no special characters & converted to lower
                    clean_list = []

                    # Clean the word by removing unwanted special characters
                    for word in list_data:
                        t = re.sub('[.?;,"]', "", word.lower())
                        # Remove all the unwanted standard stop words and user defined stop words
                        if t not in self.stop_words and word not in [',', '.', '?', '_', ';', '"', "'", ';', ':']:
                            clean_list.append(t)

                    if len(clean_list) > 0:
                        # Tag the words with their part of speech
                        pos_tag_tuple = pos_tag(clean_list)
                        refined_pos_tag_tuple = self.filter_pos(pos_tag_tuple)
                        if (refined_pos_tag_tuple is not None) and (len(refined_pos_tag_tuple) != 0):
                            # Each sentence will get a set of words eligible to be changed
                            master_data.append((sentence, refined_pos_tag_tuple,period_flag))
                    else:
                      if sentence is not " " or sentence is not "":
                        master_data.append((sentence,None,period_flag))
            print(f"Master list : {master_data}")
            
            # For each eligible word, derive the synonyms
            for _, tuple_data,_ in master_data:
              if tuple_data is not None:
                self.generate_synonyms(tuple_data)

            print(f"Dict = {self.synonym_dict}")

            # refined_sentence = ""
            refined_sentence_list = []

            # Indicating the line number
            number = 0
            
            '''
            Iterate over each sentence and reduce to lower or higer CEFR Level
            '''
            dict_with_sentence_data = {}

            list_to_save_tracker_words = []

            for sentence, tuple_data,period_flag in master_data:
                if to_reduce:
                    '''
                    When CEFR has to be lowered, reach  this
                    '''
                    print(f"Initial Sentence - {sentence}")

                    if period_flag is False:
                      '''
                      Indicates a false period, store data until real
                      '''
                      list_to_save_tracker_words.append((sentence,tuple_data))
                      
                    else:
                      '''
                      Indicates a real period, utilise the stored data
                      '''
                      local_list = []
                      list_tuple = []
                      # retrieve Stored data if any
                      for stored_sentence,tuple_info in list_to_save_tracker_words:
                        local_list.append(stored_sentence)
                        if tuple_info is not None:
                          list_tuple.extend(tuple_info)

                      # Store incoming data
                      local_list.append(sentence)
                      if tuple_data is not None:
                        list_tuple.extend(tuple_data)

                      list_to_save_tracker_words=[]
                      refined_sentence,trace = self.refine_sentence_to_lower(" ".join(local_list), list_tuple)
                      added_full_stop = refined_sentence+"."
                      print(f"Refined Sentence - {added_full_stop}")
                      refined_sentence_list.append(refined_sentence)

                    if period_flag==True:
                      dict_with_sentence_data[f"sentence_{number}"] = trace
                      number+=1

                elif to_increase:
                    '''
                    When CEFR has to be improved, reach  this
                    '''
                    refined_sentence = self.refine_sentence_to_upper(sentence, tuple_data)
                    refined_sentence_list.append(refined_sentence)

            # If the sentence were stored and not recovered because we never
            # encountered another True period sentence
            local_list = []
            list_tuple = []
            if len(list_to_save_tracker_words)>0:        
              for stored_sentence,tuple_info in list_to_save_tracker_words:
                local_list.append(stored_sentence)
                if tuple_info is not None:
                  list_tuple.extend(tuple_info)
              
              refined_sentence,trace = self.refine_sentence_to_lower(" ".join(local_list), list_tuple)
              
              if period_flag==True:
                added_full_stop = refined_sentence+"."
                # print(f"Refined Sentence - {added_full_stop}")
                refined_sentence_list.append(refined_sentence)
              else:
                refined_sentence_list.append(refined_sentence)
            
            print(f"Checking Grammer... for {len(refined_sentence_list)}  sentences ")
            print(f"Checking List - {refined_sentence_list}")

            '''
            Perform Grammatical Error Checking
            '''
            self.grammer_checked_list = set()
            self.grammer_check(refined_sentence_list)

            '''
            Since operation is performed by thread, sentence order is scatterd,
            sort the sentences based on their positional index
            '''
            sorted_grammer_checked_sentences = sorted(self.grammer_checked_list,key=lambda x:x[1])
            
            list_to_hold_sorted_data = []
            
            for sentence,_ in sorted_grammer_checked_sentences:
              list_to_hold_sorted_data.append(sentence)
            print(list_to_hold_sorted_data)

            # Variable which contains all the complete page info after Reduction/Increase & Grammer Check
            reduced_checked_page_data = "".join(list_to_hold_sorted_data).strip().replace(".",'')
            
            '''
            Code to identify correct para breaks based on the information stored
            '''
            altered_token_list = reduced_checked_page_data
            
            para_dictionary = {}

            '''
            To understand the number of sentences in the 
            paragraph, we use the below
            '''
            for para_number,_,character_width,is_line_special,break_index in para_break:
              '''
              If Character width is 0, then it means it was previously
              a \n or \t
              '''
              
              if para_dictionary.get(para_number) is None:
                para_dictionary[para_number] = 1
              elif character_width>0:
                value = para_dictionary.get(para_number)
                para_dictionary[para_number] = 1+value

            print(f"Paragraph Break Dict is {para_dictionary}")

            '''
            String for json response
            '''
            marked_string = []

            '''
            Re pack sentences for pdf, based on the 
            pre-determined sentence count in a 
            paragraph
            '''
            for index,value in para_dictionary.items():
              '''
              This is a special case for PyMuPdf - Take care
              '''
              # if index>0 and index<len(para_dictionary):

              content = list_to_hold_sorted_data[:value]
              self.re_pack_sentence(index,content)
              # Mark \n and \t for the final json response
              marked_string.append(f"{''.join(content)}\n \t ")
              # Remove the unused data from list
              list_to_hold_sorted_data = list_to_hold_sorted_data[value:]   
            return {"title":self.title,"author":self.author,"sentence_list":dict_with_sentence_data,"formatted_text":"".join(marked_string)}
        else:
            return None

    def grammer_check(self,refined_sentence_list):
      '''
      Send a list of sentences which needs to be checked
      1. Prepare 3 Threads
      2. Assign Task
      3. Fire
      4. use Join to ensure Main thread doesn't complete
      '''
      size = int(len(refined_sentence_list)/3)
     
      t1 = threading.Thread(target=self.parser_func, args=(refined_sentence_list[:size],False,0,)) 
      t2 = threading.Thread(target=self.parser_func, args=(refined_sentence_list[size:(size*2)],True,size,))
      t3 = threading.Thread(target=self.parser_func, args=(refined_sentence_list[(size*2):],True,size*2,)) 
      # Start Threads
      t1.start()
      t2.start()
      t3.start()
      # Ensure All threads finish for main thread to finish
      t1.join()
      t2.join()
      t3.join()


    def parser_func(self,list_data,to_add=False,multip=1):
      '''
      This function is handled by thread.
      In order to maintain the actual ordering of the sentences
      in the paragraph, we use index 
      '''
      size = len(list_data)
      for index,sent in enumerate(list_data):
        val = self.parser.parse(sent).get('result')+"."
        if to_add == False:
          # Meaning, index starts from 0,1, .. 
          # Started by first thread
          self.grammer_checked_list.add((val.strip(),index))
        else:
          # Meaning, started by either 2 or 3 thread
          # Manipulate index accordingly
          self.grammer_checked_list.add((val.strip(),(multip)+index))

    def filter_pos(self, pos_tag_tuple=None):
      '''
      This function filter's out the pos_tag we would require to check
      '''
      if pos_tag_tuple is not None and (len(pos_tag_tuple) != 0):
        modified_tuple_list = []
        for word, pos in pos_tag_tuple:
          if pos not in self.pos_to_exclude:
            m_pos = self.pos_converter.get(pos)
            if m_pos is not None and (str(word).startswith("'") is False):
              modified_tuple_list.append((word.replace('[,.?;]', ""), m_pos))
        return modified_tuple_list

    def re_pack_sentence(self, index_to_change, list_data):
      '''
      This function is utilised once the 
      1. Sentence is reduced/increased
      2. Grammer checked
      3. Ready for adding text to new PDF
      '''
      string_mod = " ".join(list_data)
      to_change = list(self.page_block[index_to_change])
      to_change[4] = string_mod
      self.page_block[index_to_change] = tuple(to_change)

    def create_new_pdf(self, name_of_file="modified_data"):
      '''
      Use this function to write a pdf,
      1. Always ensure self.page_blocks is populated before hand
      We Call this after self.repack in iterate_through_paragraph() 
      '''
      new_doc = fitz.open()
      page = new_doc.newPage()
      for value in self.page_block:
        rectangle = fitz.Rect(value[0], value[1], value[2], value[3])
        wr = fitz.TextWriter(page.rect)
        rect = fitz.Rect(rectangle)
        # Data determined and hard coded from the PDF
        fon = fitz.Font(fontname='URWPalladioL', fontfile="/content/urw-palladio-l-roman.ttf")
        wr.fillTextbox(rect=rect, text=value[4], fontsize=13.9, align=fitz.TEXT_ALIGN_JUSTIFY, font=fon,
                           lineheight=1.21)
        rect.y0 = wr.last_point.y
        wr.writeText(page)
        # Save your new file
      file_name = name_of_file.split(".")[0] +"_reduced"+".pdf"
      new_doc.save(file_name)  

    def generate_synonyms(self, refined_pos_tag_tuple):
      '''
      This method generates synonyms for the requested word list
      Value is populated in the global dict for synonyms and can be accessed from there
      Not Thread SAFE
      '''
      for word_to_check, part_of_speech in refined_pos_tag_tuple:
        cefr_level_word = cefr_level.getCefr(word_to_check, part_of_speech)
        if cefr_level_word is not None and cefr_level_word != 'A':
        # Check if the synonym list is not already present
          if word_to_check not in self.synonym_dict:
          # Get the synonyms list
            synonym_list = synonym_class.retrieveSynsByPos(word_to_check, part_of_speech)
            if len(synonym_list) != 0:
              # Split the synonyms list as A,B and C cefr level lists
              list_a, list_b, list_c = self.get_seperated_synonyms(synonym_list, part_of_speech)
              # Store that list as a tuple with (A_CEFR_LIST,B_CEFR_LIST,C_CEFR_LIST)
              self.synonym_dict[word_to_check] = (list_a, list_b, list_c)

    def get_seperated_synonyms(self, synonym_list, part_of_speech):
      '''
      This ensures the list of synonyms is seperated as
      list for cefr A, list for cefr B and list for cefr C
      '''
      list_for_a = []
      list_for_b = []
      list_for_c = []
      for word in synonym_list:
        refined_word = re.sub('[.?;,"]', "", word.lower())
        lv = cefr_level.getCefr(refined_word, part_of_speech)
        if lv is not None:
          if lv == 'A':
            list_for_a.append(refined_word)
          elif lv == 'B':
            list_for_b.append(refined_word)
          else:
            list_for_c.append(refined_word)
      return list_for_a, list_for_b, list_for_c

    def refine_sentence_to_lower(self, sentence, tuple_data={}):
        '''
        Main method to ensure words in sentences are reduced to lowest possible
        level 
        '''
        words_with_score_list = []
        new_sentence = copy.deepcopy(sentence)
        track_log = {}

        for word, part_of_speech in tuple_data:
            
            refined_main_word = re.sub('[.?;,"]', "", word.lower())
            main_word_cefr_level = cefr_level.getCefr(refined_main_word, part_of_speech)

            if (main_word_cefr_level is not None) and (word in self.synonym_dict):
                target_cefr = []
                # If Main word is B: Eligible ones are A or B
                if main_word_cefr_level == 'B':
                  target_cefr.append('A')
                else:
                  # If Main word Cefr is C, eligible ones are A,B,C
                  target_cefr.append('A')
                  target_cefr.append('B')
                
                # Get Synonyms
                synonym_list = CustomReader.synonym_dict.get(word)
                
                # Get index for the requested word
                index = str(re.sub('[.?;,"]', "", new_sentence.lower())).split().index(word)
                
                # To create a list of word_iteration chart, use this tracker
                tracker_key = f"index_{index}"
                
                # Clear the Global word tracker at each instance
                # Contains set of all words for a sentence
                self.word_tracker = list()

                # Adds original word data to the first position in "childrens" key
                self.word_tracker.append({"word":word,"cefr":main_word_cefr_level,"sentence":new_sentence})
                
                # Request to get the highest score from
                # Set of A,B and C Cefr that is generated for each word
                word_object = self.get_highest_score_word_tuple(new_sentence, word, synonym_list, index, part_of_speech)
                
                # Contains the final selected word before return
                qualified_word = {}

                if word_object is not None:

                  if word_object.synonym_cefr in target_cefr:
                    '''
                    Found eligible word, return
                    '''
                    qualified_word = self.create_dict(word_object)
                    new_sentence = word_object.sentence
                  else:
                    # Need to parse down  levels to figure out a lower level cefr word
                    self.initial_highest_score = word_object.score
                    
                    # To ensure, already checked word is not checked again
                    self.word_tracker.append(self.create_dict(word_object))
                    newlist = [word_object.synonym_word]

                    # Recurse though lower levels
                    word_instance =  self.level_down(word_object,part_of_speech,target_cefr,1,newlist)
                    
                    if word_instance.synonym_cefr >= main_word_cefr_level:
                      '''
                      Received a higher or equal to word
                      try to fetch the lowest CEFR word generated by recursion
                      '''
                      word_instance = self.get_a_sentence(word_instance,main_word_cefr_level)
                      
                      if(word_instance.synonym_cefr > main_word_cefr_level):
                        # Received a higher instance word, so use the main word itself
                        if word_instance.synonym_word not in newlist:
                          self.word_tracker.append(self.create_dict(word_instance))
                        new_sentence = sentence
                        qualified_word = {"word":word,"cefr":main_word_cefr_level,"sentence":new_sentence}
                      else:
                        # Use the received word
                        qualified_word = self.create_dict(word_instance)
                        new_sentence = word_instance.sentence
                    
                    else:
                      qualified_word = self.create_dict(word_instance)
                      new_sentence = word_instance.sentence
                  
                  # Clear the Priority queue for next word from
                  # the main sentence
                  self.set_a = queue.PriorityQueue()
                  self.set_b = queue.PriorityQueue()
                  self.set_c = queue.PriorityQueue()
                  
                  if len(self.word_tracker) > 0:
                    qualified_word["children"] = self.word_tracker
                  track_log[tracker_key] = qualified_word

        return (new_sentence,track_log)

    def create_dict(self,word_object):
      '''
      {
        "word":
        "score"
        "cefr":
        "sentence":
      }
      '''
      qualifier={}
      qualifier["word"] = word_object.synonym_word
      qualifier["score"] = word_object.score
      qualifier["cefr"] = word_object.synonym_cefr
      qualifier["sentence"] = word_object.sentence
      return qualifier

    def get_a_sentence(self,obj,cefr_level):
      '''
      Check the lower cefr sets to find the lowest cefr level word
      if C - > get set B or set A
      if B - > get lowest of set A
      '''
      score_1 = 0.00
      score_2 = 0.00

      if cefr_level == 'C':
        if len(self.set_a.queue)!=0:
          word_a = self.set_a.queue[0]
          score_1 = word_a.score
        if len(self.set_b.queue)!=0:
          word_b = self.set_b.queue[0]
          score_2 = word_b.score
        if (score_1==score_2) and score_1 == 0.00:
          return obj
        elif score_1==score_2:
          return word_a
        elif score_1>score_2:
          return word_a
        else:
          return word_b
      else:
        if len(self.set_a.queue)!=0:
          word_a = self.set_a.queue[0]
          score_1 = word_a.score
        if score_1 == 0.00:
          return obj
        else:
          return word_a

    def level_down(self,word_obj,pos,target_cefr,level,list_of_existing_words):
      if (level == self.max_allowed_levels) or ((word_obj.score == self.initial_highest_score) and level>1):
        return word_obj
      else:
        word = word_obj.synonym_word
        self.generate_synonyms([(word,pos)])
        syn_list = self.synonym_dict.get(word)

        if syn_list is None or (len(syn_list)==0):
          return word_obj
        else:
          index = str(re.sub('[.?;,"]', "", word_obj.sentence.lower())).split().index(word)  
          word_object = self.get_highest_score_word_tuple(word_obj.sentence, word, syn_list, index, pos)
          
          if word_object is not None:
            if word_object.synonym_cefr in target_cefr:
              return word_object
            
            else:
              
              if word_object.synonym_word not in list_of_existing_words:
                self.word_tracker.append(self.create_dict(word_object))
                list_of_existing_words.append(word_object.synonym_word)
                return self.level_down(word_object,pos,target_cefr,level+1,list_of_existing_words)
              else:
                return word_obj
          else:
            return word_obj

    def refine_sentence_to_upper(self, sentence, tuple_data):
      # TODO : Fill Later
        return sentence

    def get_top_one(self, new_sentence, word, list_data, index, part_of_speech):
        '''
        Checks the synmantic scores of the synonym words
        Returns the highest from CEFR_A,B and C
        '''
        acceptable_synonyms_list = semantic_class.checkSynPos(new_sentence, word, list_data, index,
                                                              part_of_speech)

        if len(acceptable_synonyms_list) != 0:
            return semantic_class.calcSemanticScore(new_sentence, word, acceptable_synonyms_list, index,part_of_speech)
        else:
            return None

    def get_highest_score_word_tuple(self,new_sentence, word, synonym_list, index, part_of_speech):
      '''
      Performs the Priority Queue function
      Orders the highest score element to first as and when data is queued
      
      Return the top scoring value out of Cefr_set A,B and C
      '''
      queu = queue.PriorityQueue()
      value_a = self.get_top_one(new_sentence, word, synonym_list[0], index, part_of_speech)
      value_b = self.get_top_one(new_sentence, word, synonym_list[1], index, part_of_speech)
      value_c = self.get_top_one(new_sentence, word, synonym_list[2], index, part_of_speech)
                
      if value_a is not None:
        word_new = Word(value_a[0],value_a[1],'A',value_a[2],word)
        queu.put(word_new)
        self.set_a.put(word_new)
      if value_b is not None:
        word_new = Word(value_b[0],value_b[1],'B',value_b[2],word)
        queu.put(word_new)
        self.set_b.put(word_new)
      if value_c is not None:
        word_new = Word(value_c[0],value_c[1],'C',value_c[2],word)
        queu.put(word_new)
        self.set_c = word_new
                
      if len(queu.queue)!=0:
        return queu.queue[0]           

Main Code To call the PDF Reader

In [ ]:
cefr_level = Cefr()
synonym_class = SynRetriever()
semantic_class = SemanticCheck()
class ConvertPdf:

  def __init__(self,low_score = 97.00,required_level = 'A'):
    '''
    Alter this value for a score that is considered as low
    '''
    self.low_score = low_score
    self.required_level = required_level
    self.json_string = {}

  def convert(self,name_of_file=None):
    start_time = time.time()
    
    if name_of_file is not None:
      file_path = name_of_file
      # This will unpack the PDF and initialises the required variable
      custom = CustomReader(file_path_to_pdf=file_path, page_number=0,
                                  stop_word_list=[CustomReader.stop_words_subs, CustomReader.stop_words_conj])
      # Perform the reduction on the unpacked data
      json_data = custom.iterate_over_paragraph(to_reduce=True)
      
      if json_data is not None:
        custom.create_new_pdf(file_path)
        self.json_string = json_data

      with open('/content/json_data.json', 'w') as fout:
        json.dump(self.json_string, fout)

      end_time = time.time()
      print(f"Time taken to Perform task {end_time - start_time} (secs)")    
    else:
      print("Received None filename, Please provide a file name")

100%|██████████| 461M/461M [01:06<00:00, 6.94MB/s]


# Code to install Ngrok
Use this to install ngrok which helps in tunneling local requests

In [ ]:
!pip install flask-ngrok
!pip install flask_cors

In [ ]:
!pip install pyngrok
!ngrok authtoken ***REMOVED-NGROK-TOKEN***

  Created wheel for pyngrok: filename=pyngrok-5.0.1-cp36-none-any.whl size=18822 sha256=805e3cd631508cc42bc47cb9377a3639f67986a1299f00012b753e6a368fd71b
  Stored in directory: /root/.cache/pip/wheels/94/01/05/d39efb8f6b40a411354b4168ca9dda99e6f8d586e458e97551
Successfully built pyngrok
Authtoken saved to configuration file: /root/.ngrok2/ngrok.yml


# Code to run flask and connect with Google Collab Code

In [ ]:
import os
import base64
from flask_ngrok import run_with_ngrok
from flask import Flask,request,url_for,jsonify,redirect,render_template,send_from_directory,send_file
from google.colab import files
from flask_cors import CORS

# Local directory in Google Collab
app = Flask(__name__,template_folder = "/content")
run_with_ngrok(app)
CORS(app)

file_name = ""


@app.route('/',methods=["POST","GET"])
def index():
  if request.method == "POST":
    file_data = request.files['file']
    req_data = file_data.read()
    file_name = file_data.filename

    with open('/content/'+file_name, 'wb') as fout:
      fout.write(req_data)
      c = ConvertPdf()
      c.convert(file_name)
      return redirect(url_for("bookname",bk=file_name))
  else:
    return render_template('first_page.html')

@app.route('/download',methods=["POST","GET"])
def download():
  if request.method == "POST":
    r= send_from_directory(directory='/content',
                           filename= file_name+"_reduced",
                           mimetype='application/pdf')
    return r

@app.route('/retrieve',methods=["POST","GET"])
def retreive():
  req_data = request.get_json()['request']
  byteData = req_data['bytes']
  f_name = req_data["filename"]

  if request.method == 'POST':
    with open('/content/'+f_name, 'wb') as fout:
      pdfByteData = base64.b64decode(byteData)
      fout.write(pdfByteData)
      c = ConvertPdf()
      c.convert(f_name)
      return c.json_string

@app.route('/<bk>',methods=["POST","GET"])
def bookname(bk):
  if request.method == "POST":
    # print(bk)
    name=bk.split(".pdf")[0]
    r= send_file(f"{name}_reduced.pdf", as_attachment=True)
    return r
  else:
    return render_template('second_page.html')
  

# Code to test the implementation

In [ ]:
# Run on a server - Requires Uploading a PDF 
# app.run()

#Run locally - Ensure the requested file in already present in the local /content folder
c = ConvertPdf()
c.convert("Treasure Island"+".pdf")